In [1]:
import pandas as pd
import numpy as np
import pickle

In [2]:
def run_scoring_pipeline():
    print("Starting Scoring Pipeline...")
    
    # 1. Load the engineered data
    # In production, this would be fresh data coming from the live databases.
    # For this exercise, we score all historical employees.
    df = pd.read_csv('engineered_data.csv')
    
    # Extract employee_id and the features used by the model
    employee_ids = df['employee_id']
    X_score = df.drop(columns=['employee_id', 'high_friction'])
    
    # 2. Load the trained final model
    with open('best_model.pkl', 'rb') as f:
        model = pickle.load(f)
        
    # 3. Generate Predictions and Probabilities
    # We want the probability of class 1 (high_friction)
    probs = model.predict_proba(X_score)[:, 1]
    
    # Convert to a 0-100 risk score
    risk_scores = (probs * 100).round(1)
    
    # 4. Create the final output table with percentiles for dynamic recalibration
    p80 = np.percentile(risk_scores, 80)
    p50 = np.percentile(risk_scores, 50)
    
    risk_table = pd.DataFrame({
        'employee_id': employee_ids,
        'risk_score': risk_scores,
        'risk_level': ['High' if score >= p80 else 'Medium' if score >= p50 else 'Low' for score in risk_scores]
    })
    
    # Merge some basic info for the dashboard (like Department) from master_data
    master = pd.read_csv('master_data.csv')
    dashboard_table = risk_table.merge(master[['employee_id', 'Department', 'JobRole']], on='employee_id', how='left')
    
    print("\nRisk Table Sample:")
    print(dashboard_table.head())
    
    print("\nRisk Level Distribution:")
    print(dashboard_table['risk_level'].value_counts())
    
    # 5. Export for Dashboard consumption
    dashboard_table.to_csv('risk_scores.csv', index=False)
    print("\nSaved 'risk_scores.csv'. Pipeline complete.")

In [3]:
if __name__ == '__main__':
    run_scoring_pipeline()

Starting Scoring Pipeline...



Risk Table Sample:
   employee_id  risk_score risk_level              Department  \
0            1    6.300000        Low                   Sales   
1            2   85.000000     Medium  Research & Development   
2            4   99.599998       High  Research & Development   
3            5   76.699997     Medium  Research & Development   
4            7   81.500000     Medium  Research & Development   

                 JobRole  
0        Sales Executive  
1     Research Scientist  
2  Laboratory Technician  
3     Research Scientist  
4  Laboratory Technician  

Risk Level Distribution:
risk_level
Low       735
Medium    440
High      295
Name: count, dtype: int64

Saved 'risk_scores.csv'. Pipeline complete.
